# Movie Rating Prediction

This notebook documents the end-to-end machine learning workflow for the movie rating prediction project. It includes data inspection, preprocessing, feature engineering, model selection, evaluation, and prediction examples.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_dataset
from src.preprocessing import clean_movie_dataset
from src.feature_engineering import MovieFeatureEngineer
from src.train import train_model
from src.evaluate import compute_metrics
from src.predict import predict_movie_rating

sns.set(style="whitegrid")

DF_PATH = "data/movie_dataset.csv"
df = load_dataset(DF_PATH)
print(f"Dataset shape: {df.shape}")
print("Columns:", df.columns.tolist())

df.head()

## Dataset Overview

Inspect the raw dataset, missing values, and initial column statistics.

In [ ]:
print("Missing values by column:\n", df.isna().sum())
print("\nSample rows:")
df.sample(5, random_state=42).reset_index(drop=True)

## Preprocessing

Clean the dataset and ensure the feature pipeline can parse year, duration, genre, and cast/director information.

In [ ]:
cleaned_df = clean_movie_dataset(df)
print(f"Cleaned dataset shape: {cleaned_df.shape}")
cleaned_df.head()

## Feature Engineering

Fit the reusable `MovieFeatureEngineer` and inspect engineered feature names.

In [ ]:
feature_engineer = MovieFeatureEngineer()
feature_frame = feature_engineer.fit_transform(cleaned_df.drop(columns=["Rating"]))
print(f"Engineered feature shape: {feature_frame.shape}")
print("Feature names (first 20):", feature_engineer.feature_names_[:20])
feature_frame.head()

## Model Training and Evaluation

Train the model pipeline and compute metrics on the holdout test split.

In [ ]:
artifacts = train_model(df)

best_model_name = artifacts["best_model_name"]
print(f"Best model: {best_model_name}")
print("CV results:")
for result in artifacts["cv_results"]:
    print(f"- {result['model_name']}: mean R² = {result['cv_r2_mean']:.4f}, std = {result['cv_r2_std']:.4f}")

predictions = artifacts["best_model"].predict(artifacts["X_test"])
metrics = compute_metrics(artifacts["y_test"], predictions, artifacts["X_test"].shape[1])
print("\nEvaluation metrics:")
for metric, value in metrics.items():
    print(f"- {metric}: {value:.4f}")

## Prediction Example

Use the trained pipeline for a new movie metadata input and inspect the predicted rating.

In [ ]:
new_movie = {
    "Name": "Example Film",
    "Year": "2024",
    "Duration": "125 min",
    "Genre": "Drama, Romance",
    "Votes": "1500",
    "Director": "Unknown Director",
    "Actor 1": "Unknown Actor",
    "Actor 2": "Unknown Actor",
    "Actor 3": "Unknown Actor",
}

prediction = predict_movie_rating(new_movie)
print(f"Predicted rating: {prediction.predicted_rating:.2f}")
print(f"Confidence score: {prediction.confidence_score:.3f}")
print("Top feature contributions:")
for feature, value in prediction.explanation.items():
    print(f"- {feature}: {value:.4f}")

## Notes

- The notebook uses the same pipeline modules from `src/`.
- `run_pipeline.py` is the recommended reproducible entry point.
- Optional dependencies like `xgboost`, `lightgbm`, `catboost`, and `shap` improve performance and explanation.